# TASK 5.1: THỰC THI FULL-BENCHMARK TRÊN 18 TẬP TEST HAI CHIỀU (MÁY CHỦ KAGGLE)
### ĐÁNH GIÁ TOÀN DIỆN MÔ HÌNH FATFORMER-XLA FINETUNE PHƯƠNG ÁN A (OFFLINE 100%)

> **Người thực hiện**: Thành viên C *(Training & Evaluation Lead)*  
> **Mã nhiệm vụ**: `Task 5.1` *(Milestone 4 - Tuần 5)*  
> **Môi trường thực thi**: Kaggle Server GPU (NVIDIA T4 / P100 / RTX PRO 6000, Offline 100%)  
> **Checkpoint đánh giá**: `fatformer_plan_a_final.pth` (kế thừa từ Epoch 3 kỷ lục: Val ACC 99.84%, AP 100%, AUC 100%)  
> **Mục tiêu nghiệm thu**:  
> 1. **Chiều 1 (Cross-Generator Generalization)**: Khôi phục Clean ACC $\ge 96.0\%$ trên 18 tập test (8 GANs + 10 Diffusion).  
> 2. **Chiều 2 (Degradation Robustness)**: Đạt $Q=30$ ACC $\ge 65.0\%$ và Fake ACC $\ge 50.0\%$ trên 6 biến thể suy biến vật lý.  
> 3. **Tài sản bàn giao**: Ma trận `final_full_benchmark_matrix.csv` và biểu đồ đối đầu phân giải cao `task_5.1_master_benchmark_chart.png`.

---

### HƯỚNG DẪN INPUT CẦN THÊM VÀO KAGGLE NOTEBOOK NÀY:
1. **`fatformer-code`**: Dataset chứa mã nguồn dự án.
2. **`fatformer-assets-pack`**: Dataset chứa `ViT-L-14.pt` (hoặc dataset pretrained riêng).
3. **`fatformer-plan-a-checkpoint`**: Dataset chứa checkpoint `fatformer_plan_a_final.pth` vừa huấn luyện xong.
4. **Dataset Test Benchmark**: Dataset chứa 18 tập test Clean và Degraded (dạng thư mục hoặc `.tar`).

In [ ]:
# ==============================================================================
# BƯỚC 1: THIẾT LẬP MÃ NGUỒN VÀ TÁI TẠO BPE VOCAB OFFLINE TRÊN KAGGLE
# ==============================================================================
import os
import sys
import glob
import shutil
import zipfile
import subprocess

print("=" * 80)
print("BƯỚC 1: THIẾT LẬP MÃ NGUỒN FATFORMER-XLA")
print("=" * 80)

WORK_DIR = "/kaggle/working/fatformer-xla"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# 1. Định vị mã nguồn từ Kaggle Input
code_files = glob.glob("/kaggle/input/**/train.py", recursive=True)
if not code_files:
    raise FileNotFoundError("Không tìm thấy mã nguồn (train.py) trong /kaggle/input! Hãy Add Input dataset 'fatformer-code'.")

src_dir = os.path.dirname(code_files[0])
print(f"[✓] Phát hiện mã nguồn tại: {src_dir}")
shutil.copytree(src_dir, WORK_DIR, dirs_exist_ok=True)

# 2. Xử lý BPE Tokenizer offline (Đảm bảo CLIP hoạt động không cần Internet)
clip_dir = os.path.join(WORK_DIR, "src/models/clip")
os.makedirs(clip_dir, exist_ok=True)
target_gz = os.path.join(clip_dir, "bpe_simple_vocab_16e6.txt.gz")

bpe_gz = glob.glob("/kaggle/**/bpe_simple_vocab_16e6.txt.gz", recursive=True)
bpe_txt = glob.glob("/kaggle/**/bpe_simple_vocab_16e6.txt", recursive=True)

if bpe_gz:
    shutil.copy2(bpe_gz[0], target_gz)
    print(f"[✓] Đã sao chép BPE vocab .gz: {target_gz}")
elif bpe_txt:
    subprocess.run(f"gzip -c '{bpe_txt[0]}' > '{target_gz}'", shell=True, check=True)
    print(f"[✓] Đã nén và tái tạo BPE vocab .gz: {target_gz}")

%cd {WORK_DIR}
print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")


In [ ]:
# ==============================================================================
# BƯỚC 2: QUÉT TỰ ĐỘNG CHECKPOINT, CLIP VI-T & GIẢI NÉN DỮ LIỆU TEST (NẾU CẦN)
# ==============================================================================
import os
import glob
import tarfile
import time
import torch

assert torch.cuda.is_available(), "LỖI: Chưa bật GPU! Vào Settings -> Accelerator -> Chọn GPU (T4 x2 hoặc P100)."
gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[✓] GPU khả dụng: {gpu_name} ({gpu_vram:.2f} GB VRAM) - Sẵn sàng cho Full Benchmark!")

# 1. Tự động định vị Checkpoint Plan A trong /kaggle/input hoặc /kaggle/working
ckpt_candidates = [
    "/kaggle/working/checkpoints/finetune_plan_a/fatformer_plan_a_final.pth",
    "/kaggle/working/checkpoints/finetune_plan_a/model_best.pth",
] + glob.glob("/kaggle/input/**/*fatformer*plan_a*.pth", recursive=True) \
  + glob.glob("/kaggle/input/**/*model_best*.pth", recursive=True) \
  + glob.glob("/kaggle/input/**/checkpoint/*.pth", recursive=True)

CKPT_FINAL = None
for c in ckpt_candidates:
    if os.path.isfile(c) and os.path.getsize(c) > 500 * (1024**2):  # > 500MB
        CKPT_FINAL = c
        break

if not CKPT_FINAL:
    raise FileNotFoundError("Chưa tìm thấy checkpoint Plan A (.pth) trong /kaggle/input! Hãy Add Input dataset 'fatformer-plan-a-checkpoint'.")

# 2. Tự động định vị CLIP ViT-L-14.pt
clip_candidates = glob.glob("/kaggle/**/ViT-L-14.pt", recursive=True)
if not clip_candidates:
    raise FileNotFoundError("Chưa tìm thấy ViT-L-14.pt! Hãy kiểm tra dataset fatformer-assets-pack.")
CLIP_PATH = clip_candidates[0]

# 3. Tự động định vị dữ liệu Test Clean & Degraded
DATA_LOCAL = "/kaggle/working/dataset_local"
os.makedirs(DATA_LOCAL, exist_ok=True)

# Hàm giải nén an toàn nếu dữ liệu còn ở dạng file .tar
def unpack_if_tar(tar_name, target_dir):
    tars = glob.glob(f"/kaggle/input/**/{tar_name}", recursive=True)
    if tars:
        tar_path = tars[0]
        print(f"[*] Đang giải nén {tar_name} sang {target_dir}...")
        t0 = time.time()
        with tarfile.open(tar_path, 'r') as tf:
            tf.extractall(target_dir)
        print(f"  [✓] Hoàn tất {tar_name} trong {time.time() - t0:.1f}s!")

# Tự động phát hiện thư mục test Clean đã có sẵn hoặc giải nén từ .tar
clean_dirs = glob.glob("/kaggle/input/**/test_benchmark_gans", recursive=True) \
             + glob.glob("/kaggle/input/**/test_clean", recursive=True)
if clean_dirs:
    CLEAN_ROOT = clean_dirs[0]
else:
    unpack_if_tar("test_benchmark_gans.tar", DATA_LOCAL)
    unpack_if_tar("test_benchmark_diffusion.tar", DATA_LOCAL)
    CLEAN_ROOT = DATA_LOCAL

# Tự động phát hiện thư mục test Degraded đã có sẵn hoặc giải nén từ .tar
deg_dirs = glob.glob("/kaggle/input/**/test_degraded", recursive=True)
if deg_dirs:
    DEG_ROOT = deg_dirs[0]
else:
    unpack_if_tar("test_degraded.tar", DATA_LOCAL)
    DEG_ROOT = os.path.join(DATA_LOCAL, "test_degraded")

# Thiết lập thư mục output
OUTPUT_DIR = "/kaggle/working/output_task_5.1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n--- TỔNG KẾT KIỂM TOÁN TÀI NGUYÊN TASK 5.1 (KAGGLE) ---")
print(f" • Checkpoint đánh giá:  {CKPT_FINAL} ({os.path.getsize(CKPT_FINAL)/(1024**3):.2f} GB)")
print(f" • Backbone CLIP:        {CLIP_PATH} ({os.path.getsize(CLIP_PATH)/(1024**2):.2f} MB)")
print(f" • Thư mục Test Clean:   {CLEAN_ROOT}")
print(f" • Thư mục Test Degrade: {DEG_ROOT}")
print(f" • Thư mục lưu Output:   {OUTPUT_DIR}")


In [ ]:
# ==============================================================================
# BƯỚC 3: CHIỀU 1 - FULL BENCHMARK CLEAN TRÊN 18 TẬP TEST (TOOLS/FULL_EVAL.PY)
# ==============================================================================
OUTPUT_CLEAN_CSV = os.path.join(OUTPUT_DIR, "task_5.1_clean_results.csv")
OUTPUT_CLEAN_MD = os.path.join(OUTPUT_DIR, "task_5.1_clean_report.md")

cmd_clean = (
    f"python tools/full_eval.py "
    f"--checkpoint '{CKPT_FINAL}' "
    f"--clip_path '{CLIP_PATH}' "
    f"--test_root '{CLEAN_ROOT}' "
    f"--num_vit_adapter 8 "
    f"--use_srm "
    f"--use_gating "
    f"--batch_size 32 "
    f"--num_workers 4 "
    f"--output_csv '{OUTPUT_CLEAN_CSV}' "
    f"--output_markdown '{OUTPUT_CLEAN_MD}'"
)

print("[*] Đang thực thi Chiều 1: Clean Benchmark trên 18 tập test SẠCH...")
!{cmd_clean}


In [ ]:
# ==============================================================================
# BƯỚC 4: CHIỀU 2 - DEGRADED ROBUSTNESS TRÊN 6 BIẾN THỂ (TOOLS/EVAL_DEGRADED.PY)
# ==============================================================================
OUTPUT_DEG_CSV = os.path.join(OUTPUT_DIR, "task_5.1_degraded_results.csv")
OUTPUT_DEG_MD = os.path.join(OUTPUT_DIR, "task_5.1_degraded_report.md")

# Lấy mẫu khoa học 500 ảnh/subset (250 real + 250 fake) tương tự Milestone 1 để tối ưu thời gian
cmd_deg = (
    f"python tools/eval_degraded.py "
    f"--checkpoint '{CKPT_FINAL}' "
    f"--clip_path '{CLIP_PATH}' "
    f"--degraded_root '{DEG_ROOT}' "
    f"--variants all "
    f"--num_vit_adapter 8 "
    f"--samples_per_subset 500 "
    f"--batch_size 32 "
    f"--num_workers 4 "
    f"--use_srm "
    f"--use_gating "
    f"--clean_csv '{OUTPUT_CLEAN_CSV}' "
    f"--output_csv '{OUTPUT_DEG_CSV}' "
    f"--output_markdown '{OUTPUT_DEG_MD}'"
)

print("[*] Đang thực thi Chiều 2: Degraded Benchmark trên 6 biến thể suy biến vật lý...")
!{cmd_deg}


In [ ]:
# ==============================================================================
# BƯỚC 5: HỢP NHẤT MA TRẬN MASTER BENCHMARK 2 CHIỀU
# ==============================================================================
import pandas as pd
from IPython.display import display

MASTER_CSV = os.path.join(OUTPUT_DIR, "final_full_benchmark_matrix.csv")

dfs = []
if os.path.exists(OUTPUT_CLEAN_CSV):
    clean_df = pd.read_csv(OUTPUT_CLEAN_CSV)
    clean_df["variant"] = "clean"
    dfs.append(clean_df)

if os.path.exists(OUTPUT_DEG_CSV):
    deg_df = pd.read_csv(OUTPUT_DEG_CSV)
    dfs.append(deg_df)

if dfs:
    master_df = pd.concat(dfs, ignore_index=True)
    master_df.to_csv(MASTER_CSV, index=False)
    print(f"[✓] Đã tạo thành công Ma trận Master Benchmark: {MASTER_CSV} ({len(master_df)} hàng)!")
    display(master_df.head(25))
else:
    print("[!] Chưa tìm thấy kết quả từ Bước 3 và Bước 4.")


In [ ]:
# ==============================================================================
# BƯỚC 6: TRỰC QUAN HÓA TOÀN DIỆN VÀ VẼ BIỂU ĐỒ ĐỐI ĐẦU PHÂN GIẢI CAO
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if os.path.exists(MASTER_CSV):
    df = pd.read_csv(MASTER_CSV)
    print("\n--- TỔNG KẾT THEO TỪNG BIẾN THỂ TRÊN 18 TẬP TEST ---")
    summary_rows = df[df["subset"] == "MEAN_OVERALL"]
    if summary_rows.empty:
        summary_rows = df.groupby("variant")[["acc", "f_acc", "r_acc"]].mean().reset_index()
    display(summary_rows)

    # Vẽ biểu đồ Master Comparison
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, ax = plt.subplots(figsize=(12, 6), dpi=300)

    variants = summary_rows["variant"].tolist()
    acc_vals = summary_rows["acc"].tolist()
    fake_vals = summary_rows["f_acc"].tolist()

    x = np.arange(len(variants))
    width = 0.35

    r1 = ax.bar(x - width/2, acc_vals, width, label='Overall Accuracy (%)', color='#2ecc71', edgecolor='black')
    r2 = ax.bar(x + width/2, fake_vals, width, label='Fake Accuracy (%)', color='#3498db', edgecolor='black')

    ax.set_ylabel('Độ chính xác (%)', fontsize=12, fontweight='bold')
    ax.set_title('HIỆU NĂNG FATFORMER-XLA PLAN A TRÊN 18 TẬP TEST HAI CHIỀU\n(Clean vs 6 Biến thể Suy biến Vật lý)', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(variants, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 115)
    ax.legend(loc='upper right', frameon=True, fontsize=11)

    for rect in list(r1) + list(r2):
        h = rect.get_height()
        ax.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')

    chart_path = os.path.join(OUTPUT_DIR, "task_5.1_master_benchmark_chart.png")
    plt.tight_layout()
    plt.savefig(chart_path, dpi=300)
    plt.show()
    print(f"[✓] Biểu đồ đã lưu thành công tại: {chart_path}")


In [ ]:
# ==============================================================================
# BƯỚC 7: TẠO FILELINK TẢI KẾT QUẢ VỀ MÁY TÍNH TRỰC TIẾP TRÊN TRÌNH DUYỆT
# ==============================================================================
from IPython.display import FileLink, display
import os

print("=" * 80)
print("TỔNG HỢP LINK TẢI KẾT QUẢ TASK 5.1 VỀ MÁY TÍNH:")
print("=" * 80)

rel_csv = os.path.relpath(MASTER_CSV, start="/kaggle/working")
rel_chart = os.path.relpath(os.path.join(OUTPUT_DIR, "task_5.1_master_benchmark_chart.png"), start="/kaggle/working")

if os.path.exists(MASTER_CSV):
    print("👉 Bấm link dưới đây để tải Ma trận kết quả (CSV):")
    display(FileLink(rel_csv))

if os.path.exists(os.path.join(OUTPUT_DIR, "task_5.1_master_benchmark_chart.png")):
    print("\n👉 Bấm link dưới đây để tải Biểu đồ đối đầu (PNG):")
    display(FileLink(rel_chart))
